# Qwen3-VL-8B Hebrew OCR fine-tune (Unsloth, Colab A100)

Trains vision **and** language LoRA on the Talmud dataset at `isaacmg/talmud_finetune`.

**Before running:** Runtime → Change runtime type → **A100 GPU**. Add two Colab
secrets (key icon, left sidebar): `HF_TOKEN` (write-scoped) and `WANDB_API_KEY`.

**Run cells strictly in order.** Cells 5 and 6 are gates: they hard-fail if the
data plumbing is broken (a previous fine-tuning attempt in this project trained
for days with images never reaching the model — these gates make that
impossible to miss).

**If Colab disconnects:** reconnect and rerun from cell 1 — every step is cached
or resumable. Mid-training kills lose at most ~250 steps: cell 8 resumes from the
latest hub checkpoint automatically.

In [ ]:
# Cell 1 — installs + environment report
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
%pip install -q "unsloth[colab-new]" hf_transfer wandb

import torch
assert torch.cuda.is_available(), "No GPU — switch the runtime to A100"
import transformers, trl, unsloth
print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| trl", trl.__version__, "| unsloth", unsloth.__version__)

In [ ]:
# Cell 2 — dataset: fast resumable download, then hard count asserts
from google.colab import userdata
from huggingface_hub import login, snapshot_download
from datasets import load_dataset

login(token=userdata.get("HF_TOKEN"))

DATASET_REPO = "isaacmg/talmud_finetune"

# multi-threaded download, resumes partials, cached across cell reruns
snapshot_download(DATASET_REPO, repo_type="dataset")

train_all = load_dataset(DATASET_REPO, split="train")
smoke = load_dataset(DATASET_REPO, split="smoke")
val = load_dataset(DATASET_REPO, split="val")

crops = train_all.filter(lambda t: t == "crop_transcribe", input_columns="task")
pages = train_all.filter(lambda t: t == "page_extract", input_columns="task")
print(f"crops={len(crops)}  pages={len(pages)}  val={len(val)}  smoke={len(smoke)}")

# partial/corrupt download protection
assert len(train_all) == 17476 and len(crops) == 14696 and len(pages) == 2780
assert len(smoke) == 32 and len(val) == 340

In [ ]:
# Cell 3 — model + LoRA + resolution policy, ALL IN ONE CELL.
# State-safe: rerunning this cell always leaves `model`/`tokenizer` fully set up
# (a partial rerun of an earlier version of this cell once produced a model with
# no adapters, which is unfinetunable).
from unsloth import FastVisionModel
from transformers import AutoImageProcessor

MAX_SEQ = 12288          # native-res pages (~4.4k image tokens) + longest answers
MIN_PIX = 256 * 28 * 28  # never let dense Hebrew drop below readability
MAX_PIX = 4_500_000      # A100: full pages at native resolution

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,      # attack the vision encoder directly
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0,
    bias="none", random_state=3407,
)

# transformers 5.x: min_pixels/max_pixels are read-only properties (size may be
# a SizeDict) — REBUILD the processor instead of mutating it; the kwargs work
# on every version.
tokenizer.image_processor = AutoImageProcessor.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)

# fail fast if any of the above silently didn't take
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
lora_params = [n for n, p in model.named_parameters()
               if p.requires_grad and "lora" in n.lower()]
assert trainable > 0 and lora_params, \
    "No trainable LoRA parameters — get_peft_model did not run"
print(f"trainable params: {trainable/1e6:.1f}M ({len(lora_params)} LoRA tensors)")
print("resolution policy:", tokenizer.image_processor.size)


In [ ]:
# Cell 4 — dataset rows -> Unsloth conversation format
def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

converted_smoke = [to_conversation(smoke[i]) for i in range(len(smoke))]
print(converted_smoke[0]["messages"][0]["content"][1]["text"][:100])

In [ ]:
# Cell 5 — GATE 1: collator guardrail. Do not train if this cell fails.
# Guards the exact failure modes that killed the old Gemma pipeline:
#   (a) images not reaching the model   (b) loss on prompt/padding tokens
#   (c) silent downscaling (Unsloth's default resize="min" -> 512px thumbnails)
from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(
    model, tokenizer,
    resize="max",                       # never downscale images
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,       # mask loss to the assistant answer
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

batch = collator([converted_smoke[0], converted_smoke[1]])

# (a) images flow...
pv = batch.get("pixel_values")
assert pv is not None, "pixel_values is None — images are NOT reaching the model"
assert float(pv.abs().sum()) > 0, "pixel_values all-zero — preprocessing broken"
# ...at native resolution: smoke[0] (gemara, ~1058x2199) alone is ~9k patch rows.
# The 512px default produced only 5,148 rows for BOTH samples.
assert pv.shape[0] > 8000, (
    f"pixel_values has only {pv.shape[0]} patch rows — images are being "
    f"downscaled (the 512px default is back). Check the collator's resize arg."
)
print(f"pixel_values OK: shape={tuple(pv.shape)} (native resolution)")

# (b) labels mask down to the assistant answer
labels = batch["labels"]
unmasked = labels[0][labels[0] != -100]
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
decoded = _tok.decode(unmasked)
answer = converted_smoke[0]["messages"][1]["content"][0]["text"]
prompt = converted_smoke[0]["messages"][0]["content"][1]["text"]
assert answer[:40] in decoded, f"answer missing from labels.\nDecoded: {decoded[:200]}"
assert prompt[:40] not in decoded, "prompt found in labels — loss leaks onto the prompt"
frac = float((labels[0] != -100).float().mean())
print(f"label masking OK: {frac:.1%} of tokens in the loss (answer only)")

In [ ]:
# Cell 6 — TRL compat shim + GATE 2: overfit-8 (must reach loss < 0.15)
# The shim adapts to TRL API drift (max_seq_length->max_length rename,
# tokenizer->processing_class rename) instead of crashing on kwargs.
import inspect
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ SFTConfig on this TRL doesn't accept {sorted(dropped)} — "
              f"dropped; VERIFY none are load-bearing for this run.")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

COMMON_CFG = dict(
    bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
    remove_unused_columns=False, dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=MAX_SEQ, seed=3407, optim="adamw_8bit",
)

FastVisionModel.for_training(model)
overfit_trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=collator,
    train_dataset=converted_smoke[:8],
    args=make_sft_config(
        per_device_train_batch_size=1, gradient_accumulation_steps=1,
        max_steps=300, learning_rate=1e-4, logging_steps=25,
        lr_scheduler_type="constant", output_dir="outputs_overfit",
        report_to="none", **COMMON_CFG,
    ),
)
stats = overfit_trainer.train()
# gate on the FINAL logged loss - stats.training_loss is the mean over
# the whole run and stays high even when the end state is ~0.
final_losses = [e["loss"] for e in overfit_trainer.state.log_history if "loss" in e]
final_loss = final_losses[-1]
print(f"final logged loss {final_loss:.4f} (run mean {stats.training_loss:.4f})")
assert final_loss < 0.15, (
    f"Overfit-8 FAILED (final loss {final_loss:.3f} >= 0.15) - the pipeline cannot "
    "even memorize 8 samples. Something is broken; do NOT run full training."
)


In [ ]:
# Cell 7 — eyeball an overfit generation vs its ground truth
FastVisionModel.for_inference(model)
sample = smoke[0]
messages = [{"role": "user", "content": [
    {"type": "image"}, {"type": "text", "text": sample["question"]},
]}]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(sample["image"], input_text,
                   add_special_tokens=False, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
# compare HEAD of generation to HEAD of GT (comparing the tail of a long
# generation to the head of the GT once caused a false alarm here)
gen = tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:])[0]
print("GENERATED:", gen[:400])
print()
print("GROUND TRUTH:", sample["answer"][:400])


In [ ]:
# Cell 8 — FULL RUN: 85% crops / 15% pages, 2 epochs, W&B, hub checkpoints.
# Preemption-proof: checkpoints push to CKPT_REPO every 250 steps; rerunning
# this cell after a disconnect resumes from the latest checkpoint (optimizer
# state + schedule position preserved).
import wandb
from datasets import interleave_datasets
from huggingface_hub import list_repo_files, snapshot_download

CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-ckpt"  # private, auto-created
os.environ["WANDB_PROJECT"] = "qwen-hebrew-finetune"  # TRL reads env, not config
wandb.login(key=userdata.get("WANDB_API_KEY"))

mixture = interleave_datasets(
    [crops, pages], probabilities=[0.85, 0.15], seed=3407,
    stopping_strategy="all_exhausted",
)
# NO upfront .map(): a 7-minute preprocessing pass once died to a Colab
# disconnect before training even began. The collator below formats rows
# lazily per-batch instead (formatting_func), so training starts in seconds.
train_collator = UnslothVisionDataCollator(
    model, tokenizer,
    formatting_func=to_conversation,
    resize="max",
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

resume_dir = None
try:
    # hub_strategy="checkpoint" pushes a rolling "last-checkpoint/" folder
    # (NOT numbered checkpoint-N folders).
    files = list_repo_files(CKPT_REPO)
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(CKPT_REPO, allow_patterns="last-checkpoint/*",
                          local_dir="outputs_full")
        resume_dir = "outputs_full/last-checkpoint"
        print("resuming from last-checkpoint")
except Exception as e:
    print(f"no checkpoint repo yet ({type(e).__name__}) — fresh start")

FastVisionModel.for_training(model)
trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=train_collator,
    train_dataset=mixture,
    args=make_sft_config(
        # batch 1 x accum 8: native-resolution pages are ~4.4k image tokens;
        # raise to 2 x 4 only if memory clearly allows.
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=1e-4, warmup_ratio=0.03,
        lr_scheduler_type="cosine", weight_decay=0.01, logging_steps=10,
        save_steps=100, save_total_limit=2,  # first hub checkpoint ~17 min in
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=True,
        output_dir="outputs_full", report_to="wandb",
        run_name="colab_joint_qlora_crops85_pages15", **COMMON_CFG,
    ),
)
trainer.train(resume_from_checkpoint=resume_dir)


In [ ]:
# Cell 9 — export merged fp16 weights and push (private)
MERGED_REPO = "isaacmg/qwen3-vl-8b-hebrew-merged"
try:
    model.save_pretrained_merged("qwen3-vl-8b-hebrew-merged", tokenizer,
                                 save_method="merged_16bit")
    model.push_to_hub_merged(MERGED_REPO, tokenizer,
                             save_method="merged_16bit", private=True)
    print(f"pushed merged model -> {MERGED_REPO}")
except Exception as e:
    # fallback: push adapters only — still convertible locally
    print(f"merged export failed ({type(e).__name__}: {e}); pushing adapters instead")
    model.push_to_hub(MERGED_REPO + "-adapters", private=True)
    tokenizer.push_to_hub(MERGED_REPO + "-adapters", private=True)
    print(f"pushed adapters -> {MERGED_REPO}-adapters")

## Back on the Mac: convert to MLX and benchmark

```bash
# 8-bit MLX conversion of the merged model
.venv-mlx/bin/python -m mlx_vlm.convert \
    --hf-path isaacmg/qwen3-vl-8b-hebrew-merged \
    --mlx-path models/qwen3-vl-8b-heb-colab -q --q-bits 8

# guardrail probe + quick CER on the converted model (sequential — one MLX
# process at a time on the Mac!)
.venv-mlx/bin/python -m src.finetuning.qwen_hebrew.image_dependence_probe \
    --model models/qwen3-vl-8b-heb-colab
.venv-mlx/bin/python -m src.finetuning.qwen_hebrew.quick_eval \
    --model models/qwen3-vl-8b-heb-colab --num_samples 30 \
    --save_outputs models/qwen3-vl-8b-heb-colab/eval_outputs.jsonl

# import into LM Studio, then the full before/after benchmark
lms import models/qwen3-vl-8b-heb-colab
python -m src.datasets.evaluations.talmud_evaluation \
    --lm_studio_models qwen/qwen3-vl-8b,<lm-studio-id-of-fine-tune>
# pure-reading crop track:
python -m src.datasets.evaluations.talmud_crop_evaluation \
    --lm_studio_models qwen/qwen3-vl-8b,<lm-studio-id-of-fine-tune> --wandb
```